In [ ]:
# BLOCO 1: Instalações

# Pins e motivos:
# - av<19: o PyAV 19 removeu o argumento 'metadata_errors' de av.open(), que o faster-whisper 1.2.1
#   ainda passa. Sem o teto, toda transcrição falha com TypeError. Remover quando o faster-whisper
#   lançar versão compatível com o PyAV 19.
# - nvidia-cublas-cu12 / nvidia-cudnn-cu12: exigidas pelo ctranslate2 (faster-whisper); o
#   pré-carregamento é feito em utils/cuda_preloader.py, chamado por src/transcription_processor.py.
# Não instale numpy, torch nem WhisperX: o Colab já traz o que é preciso e eles quebram o ambiente.
!pip install -q faster-whisper "av<19" notion-client nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

print("✅ Instalação concluída. Não é necessário reiniciar a sessão.")

In [ ]:
# BLOCO 2: Montagem do Drive e Configurações da Sessão
from google.colab import drive, userdata
import os
import sys

drive.mount('/content/drive')

# ----------------------------------------------------------------------
# EDITE APENAS ESTAS VARIÁVEIS PARA CADA NOVA SESSÃO:
# ----------------------------------------------------------------------

ID_MESA = "TSO"
SESSION_NUMBER = "001"
SESSION_DATE = "2026-11-19"  # Formato YYYY-MM-DD
FORCE_UPDATE = False  # True para puxar mudanças do repo

# Deixe como None para usar a fila padrão definida em config/gemini_config.json
# Para testar um modelo pontualmente, informe o ID (ex: 'gemini-3.8-flash')
# Vale para os Blocos 4 (crônica) e 6 (log)
MODEL_OVERRIDE = None

# ----------------------------------------------------------------------

PROJECT_FOLDER = "/content/drive/MyDrive/rpg-transcription"

# Chave da Gemini API (usada pelos Blocos 4 e 6; o Bloco 3 não precisa dela)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = None
    print("⚠️ Secret 'GEMINI_API_KEY' não encontrado. Os Blocos 4 e 6 não vão rodar sem ela.")

# ----------------------------------------------------------------------
# CLONAGEM / ATUALIZAÇÃO DO REPOSITÓRIO
# ----------------------------------------------------------------------

if not os.path.exists('rpg_transcription'):
    !git clone https://github.com/ThaisMosken/rpg_transcription.git
elif FORCE_UPDATE:
    !git -C rpg_transcription pull

sys.path.append('/content/rpg_transcription')

# ----------------------------------------------------------------------
# CONFIGURAÇÃO DA SESSÃO (caminhos e glossário derivados em utils/session_helper.py)
# ----------------------------------------------------------------------

from utils.session_helper import setup_session

session = setup_session(ID_MESA, SESSION_NUMBER, PROJECT_FOLDER)

SESSION_ID = session["session_id"]
INPUT_FILE = session["input_file"]
OUTPUT_TXT_FILE = session["output_txt_file"]
CHRONICLE_OUTPUT_FILE = session["output_chronicle_file"]
WORKBENCH_FOLDER = session["workbench_folder"]
CONFIG_FOLDER = session["config_folder"]
LOG_OUTPUT_FILE = session["log_output_file"]
context_glossary = session["context_glossary"]
GLOSSARY = session["glossary_names"]

print(f"✅ Glossário para '{ID_MESA}' carregado com sucesso!")

print("Configuração de sessão concluída:")
print(f"  ID da Sessão: {SESSION_ID}")
print(f"  Entrada WAV (arquivo único; partes são detectadas automaticamente): {INPUT_FILE}")
print(f"  Saída TXT: {OUTPUT_TXT_FILE}")
print(f"  Saída Crônica: {CHRONICLE_OUTPUT_FILE}")
print(f"  Pasta de trabalho: {WORKBENCH_FOLDER}")

In [ ]:
# BLOCO 3: Execução da Transcrição (detecta partes automaticamente)
from src.transcription_processor import transcribe_session

transcribe_session(
    session_id=SESSION_ID,
    project_folder=WORKBENCH_FOLDER,
    input_file=INPUT_FILE,
    output_file=OUTPUT_TXT_FILE,
    glossary_names=GLOSSARY,
    model_precision="float16", # "int8_float16" ou "float16"
    model_name="large-v2"
)

In [ ]:
# BLOCO 4: Geração da Crônica
from src.chronicler import generate_gemini_chronicle

# Define o template a ser utilizado
# Utilize template_session0_v1 para sessão 0 e template_chronicle_v1 para sessões normais
TEMPLATE = 'template_chronicle_v1'

# Executa a função importada
if GEMINI_API_KEY:
    generate_gemini_chronicle(
        api_key=GEMINI_API_KEY,
        transcription_path=OUTPUT_TXT_FILE,
        output_path=CHRONICLE_OUTPUT_FILE,
        context_glossary=context_glossary,
        template=TEMPLATE,
        model=MODEL_OVERRIDE
    )
else:
    print("❌ ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

In [ ]:
# BLOCO 5: Publicação no Notion
import json
from utils.notion_publisher import NotionPublisher
from google.colab import userdata

with open(os.path.join(CONFIG_FOLDER, "notion_config.json"), "r") as f:
    NOTION_CONFIG = json.load(f)

try:
    publisher = NotionPublisher(
        token=userdata.get("NOTION_TOKEN"),
        databases_config=NOTION_CONFIG
    )
    
    print(f"🚀 Iniciando publicação da Sessão {SESSION_NUMBER}...")
    page_id = publisher.publish_session(ID_MESA, SESSION_NUMBER, SESSION_DATE, CHRONICLE_OUTPUT_FILE)
    print(f"✅ Sucesso! Página criada: https://notion.so/{page_id.replace('-', '')}")

except Exception as e:
    print(f"❌ Erro na integração com Notion: {e}")

In [ ]:
# BLOCO 6: Geração do Log Factual
from src.session_logger import generate_gemini_log

LOG_TEMPLATE = 'template_log_v1'

if GEMINI_API_KEY:
    generate_gemini_log(
        api_key=GEMINI_API_KEY,
        transcription_path=OUTPUT_TXT_FILE,
        chronicle_path=CHRONICLE_OUTPUT_FILE,
        output_path=LOG_OUTPUT_FILE,
        context_glossary=context_glossary,
        session_id=SESSION_ID,
        session_date=SESSION_DATE,
        template=LOG_TEMPLATE,
        model=MODEL_OVERRIDE
    )
else:
    print("❌ ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

In [ ]:
# BLOCO 7: Arquivar sessão (rode só depois de revisar tudo)
from src.session_archiver import archive_session

archive_session(
    project_folder=PROJECT_FOLDER,
    table_id=ID_MESA,
    session_id=SESSION_ID,
    batch_size=20
)